In [1]:
!pip install transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split

import evaluate

In [3]:
df = pd.read_csv("final_cleaned_dataset.csv")

In [4]:
df.head()

,text,label
0,us to search russian consulate in san francisc...,1
1,turkey seizes largest ever haul of ancient sta...,1
2,breaking hr mcmaster explains why washington p...,0
3,seventyfour percent of rhode islanders support...,0
4,adidas flawlessly trolls antigay bigots who sl...,0


In [5]:
df.shape

(53635, 2)

In [6]:
df["label"].value_counts()

,count
label,
1,27764
0,25871


In [7]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

In [8]:
train_dataset = Dataset.from_pandas(train_df)

test_dataset = Dataset.from_pandas(test_df)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [11]:
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/42908 [00:00<?, ? examples/s]

Map:   0%|          | 0/10727 [00:00<?, ? examples/s]

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
accuracy_metric = evaluate.load("accuracy")

f1_metric = evaluate.load("f1")

In [14]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

In [16]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [17]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.194580,0.203486,0.894472,0.892783
2,0.188047,0.246599,0.896430,0.897538


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10728, training_loss=0.19727764015852028, metrics={'train_runtime': 2159.4966, 'train_samples_per_second': 39.739, 'train_steps_per_second': 4.968, 'total_flos': 5683911141531648.0, 'train_loss': 0.19727764015852028, 'epoch': 2.0})

In [19]:
trainer.save_model("distilbert_model")

tokenizer.save_pretrained("distilbert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert_model/tokenizer_config.json', 'distilbert_model/tokenizer.json')

In [20]:
!ls

distilbert_model  drive  final_cleaned_dataset.csv  results  sample_data


In [21]:
!zip -r distilbert_model.zip distilbert_model

  adding: distilbert_model/ (stored 0%)
  adding: distilbert_model/model.safetensors (deflated 8%)
  adding: distilbert_model/training_args.bin (deflated 53%)
  adding: distilbert_model/config.json (deflated 49%)
  adding: distilbert_model/tokenizer_config.json (deflated 42%)
  adding: distilbert_model/tokenizer.json (deflated 71%)
